<a href="https://colab.research.google.com/github/doomguy0991/dls_c/blob/main/C1%20-%20Neural%20Networks%20and%20Deep%20Learning/Week%202/Logistic%20Regression%20as%20a%20Neural%20Network/N_Logistic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [87]:
import os

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Setting up Colab environment...")

    # 1. Clone the repository
    repo_url = "https://github.com/doomguy0991/dls_c.git"
    # Only clone if the directory doesn't exist yet
    if not os.path.exists("/content/dls_c"):
        !git clone $repo_url /content/dls_c

    # 2. Change working directory to the notebook's location so relative paths for libraries work imports
    %cd "/content/dls_c/C1 - Neural Networks and Deep Learning/Week 2/Logistic Regression as a Neural Network"

    print("Setup complete. You can now run the rest of the notebook.")
else:
    print("Running locally or out of Colab. No setup needed.")


Setting up Colab environment...
/content/dls_c/C1 - Neural Networks and Deep Learning/Week 2/Logistic Regression as a Neural Network
Setup complete. You can now run the rest of the notebook.


### 0. Theory

- $m$: The total number of training examples
- $n_x$: The number of features per example (e.g., $64 \times 64 \times 3 = 12288$).
- $\alpha$: The learning rate (how big of a step we take down the gradient).
- $X$: The input data. It contains all features for all examples. **$shape: (n_x, m)$**
- $Y$: The true labels (0 for non-cat, 1 for cat).$Y$ shape: $(1, m)$
- $w$: The weights (how much importance to give each pixel).$w$ shape: $(n_x, 1)$. can be initialized with 0's

- $b$: The bias (a single baseline number). $b$ shape: A single scalar (number).

- $Z$ and $A$ shape: $(1, m)$ -> Same shape as $Y$, because they are our predictions for every image.

- $dw$ shape: $(n_x, 1)$ -> Gradients must always match the shape of the original variable!
- $db$ shape: scalar.

---
$Z = w^T X + b$

$A = \sigma(Z)$

$\sigma(Z) = 1 / 1+e^{-Z}$

$J = -\frac{1}{m}\sum_{i=1}^mY\log(A)+(1-Y)\log(1-A)$

$dW = \frac{\partial J}{\partial w} = \frac{1}{m}X(A-Y)^T$

$dB = \frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^m (A-Y)$



In [88]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import scipy
from PIL import Image
from scipy import ndimage

%matplotlib inline

### 1. Load Dataset

In [89]:
def load_dataset():
  train_dataset = h5py.File('datasets/train_catvnoncat.h5', "r")
  test_dataset = h5py.File('datasets/test_catvnoncat.h5', "r")

  # get keys of dataset
  # print(list(train_dataset.keys()))

  # getting to know the shape of dataset by keys
  # for key in train_dataset.keys():
  #   data = train_dataset[key]
  #   print(f"{key}: shape={data.shape}, dtype={data.dtype}")

  train_set_x_orig = np.array(train_dataset["train_set_x"][:])
  train_set_y_orig = np.array(train_dataset["train_set_y"][:])

  test_set_x_orig = np.array(test_dataset["test_set_x"][:])
  test_set_y_orig = np.array(test_dataset["test_set_y"][:])

  classes = np.array(train_dataset["list_classes"][:])


  return train_set_x_orig, train_set_y_orig, test_set_x_orig, test_set_y_orig,classes


train_set_x_orig, train_set_y_orig, test_set_x_orig, test_set_y_orig,classes = load_dataset()

### 2. Reashaping the dataset

In [90]:
print ("train_set_x shape: " + str(train_set_x_orig.shape)) # (209, 64, 64, 3), 209 training exaples and each data is of (64,64,3)
print ("train_set_y shape: " + str(train_set_y_orig.shape)) # (209,)

m = train_set_x_orig.shape[0] # 209 -> # training examples
mbar = test_set_x_orig.shape[0]

# we expect X -> to be of (nx, m)  . so we flatten the train_set_x
# m = 209, nx = 64 * 64 * 3 -> shape[1] * shape[2] * shape[3]......
# so now each img has 64*64*3 features

# Flatten the images: (209, 64, 64, 3) -> (209, 12288)
# Then transpose (.T) -> (12288, 209)
X_train = train_set_x_orig.reshape(train_set_x_orig.shape[0], -1).T
X_test = test_set_x_orig.reshape(test_set_x_orig.shape[0], -1).T

# we expect Y -> to be of (1, mbar). but it is currently a 1 rank array so
Y_test = test_set_y_orig.reshape(1, mbar)
Y_train = train_set_y_orig.reshape(1, m)


print(X_train.shape)
print(Y_train.shape)
print(X_test.shape)
print(Y_test.shape)




train_set_x shape: (209, 64, 64, 3)
train_set_y shape: (209,)
(12288, 209)
(1, 209)
(12288, 50)
(1, 50)


### 3. Standardization of input values

- if the features ar too large or too small it causes Z to me too big or too samll
- as a result sigmoid produce 1 or 0 due to the large positive/ large negative values of z .
- so we need to standardize the values of X



In [91]:

X_test = X_test / 255
X_train = X_train / 255

# print(X)

### 4.Helper functions

In [92]:
# initizlizw W, B

def initialize_W_B(X):
  nx = X.shape[0]
  W = np.zeros((nx,1))  # double bracket needed
  B = 0.0


  assert(W.shape == (nx, 1))
  assert(isinstance(B, float) or isinstance(B, int))

  return W, B

In [93]:
import numpy as np
# Train the model on test data sets and determine W, B
def optimize (m, W, B, X, Y, num_iterations= 2000, learning_rate = 0.5, print_cost = False):
  costs = []

  for i in range (num_iterations):
    Z = np.dot(W.T, X) + B
    A = 1 / (1 + np.exp(-Z))
    cost_J = np.sum((Y*np.log(A) + (1-Y)*np.log(1-A)))/(-m)

    dW = np.dot(X, (A - Y).T) / m
    dB = np.sum(A - Y) / m

    assert(dW.shape == W.shape)
    assert(isinstance(dB, float) or isinstance(dB, int))

    W = W - (learning_rate * dW)
    B = B - (learning_rate * dB)

    # Record the costs
    if i % 100 == 0:
      costs.append(cost_J)

    # Print the cost every 100 training iterations
    if print_cost and i % 100 == 0:
      print ("Cost after iteration %i: %f" %(i, cost_J))

  return  W, B, costs

In [94]:
# Predict whether the label is 0 or 1 using learned logistic regression parameters (W, B)

def predict(W, B, X_test):
    # W should be in correct shape after train
    # not always required, but safe and common.
    W = W.reshape(X_test.shape[0], 1)

    Z = np.dot(W.T, X_test) + B
    A = 1 / (1 + np.exp(-Z))

    # Y_prediction = (A > 0.5) -> [[False, True, True, False]]
    # if we want 0 or 1 use typecast .astype(int)
    Y_prediction = (A > 0.5).astype(int)

    return Y_prediction


### 5. Main function

In [95]:
def main():
    # load datasets, reshape, standerdize  -> run section 1, 2, 3

    learning_rate = 0.005
    num_iterations = 2000
    print_cost = False

    m = X_train.shape[1]
    W, B = initialize_W_B(X_train)

    W , B, costs = optimize(m, W, B, X_train, Y_train,num_iterations, learning_rate, print_cost)

    Y_prediction_train = predict(W, B, X_train)
    Y_prediction_test = predict(W, B, X_test)

    # Print train/test Errors
    print("train accuracy: {} %".format(100 - np.mean(np.abs(Y_prediction_train - Y_train)) * 100))
    print("test accuracy: {} %".format(100 - np.mean(np.abs(Y_prediction_test - Y_test)) * 100))

    d = {"costs": costs,
      "Y_prediction_test": Y_prediction_test,
      "Y_prediction_train" : Y_prediction_train,
      "w" : W,
      "b" : B,
      "learning_rate" : learning_rate,
      "num_iterations": num_iterations}

    # print (d)

if __name__ == "__main__":
    main()

train accuracy: 99.04306220095694 %
test accuracy: 70.0 %
